### SCPT to VS all-in-one tool (Beta)

In [1]:
# Import packages for use:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import HTML, Layout, HBox, VBox, Dropdown, FloatText, Button, Textarea
from IPython.display import display
plt.rcParams['font.family'] = 'Times New Roman'
pd.set_option('display.max_columns', None)

### IMPORT TRAVEL TIME DATA

In [2]:
'''

read travel time data csv as dataframe, 
make sure the last row is not NAN in the .csv file 
and unit of depth is meter and unit of travel time is millisecond.
change the name to the target input file. 

'''

# TT_DATA = pd.read_csv('downhole_KS.csv')
# TT_DATA = pd.read_csv('SCPT_WYE_77_tt.csv')
TT_DATA = pd.read_csv('AVN_1_TT.csv')


TT_depth = np.asarray(TT_DATA['depth (m)'])
Meas_TT = np.asarray(TT_DATA['traveltime (ms)'])

### IMPORT CPT DATA

In [3]:
'''

read CPT data csv as dataframe, 
make sure the last row is not NAN in the .csv file 
and unit of depth is meter and unit of qt and fs are MPa.
change the name to the target input file. 

'''

# CPT_DATA = pd.read_csv('SCPT_WYE_77.csv')
CPT_DATA = pd.read_csv('AVN_1.csv')

CPT_depth = np.asarray(CPT_DATA['depth (m)'])
CPT_qt = np.asarray(CPT_DATA['qt (MPa)'])
CPT_fs = np.asarray(CPT_DATA['fs (MPa)'])
CPT_qc = np.asarray(CPT_DATA['qc (MPa)'])

valid_mask = ~np.isnan(CPT_depth) & ~np.isnan(CPT_qt) & ~np.isnan(CPT_fs)

CPT_depth = CPT_depth[valid_mask]
CPT_qt = CPT_qt[valid_mask]
CPT_fs = CPT_fs[valid_mask]
CPT_qc = CPT_qc[valid_mask]

# Enter depth of groundwater table in meter, if the groundwater table is not encountered, please enter a value larger than the maximum depth of CPT data.
print (np.max (CPT_depth))
dGWT = 0

30.55


### Calculate Robertson

In [5]:
from National_Model_V1 import get_CPT

fz, Ic_NM, Qtncs_NM, CPT_depth, Qtn_inv, Ic_inv, Qtn_Robertson, Ic_Robertson, kc = get_CPT(CPT_depth, CPT_qt, CPT_fs, dGWT)
Vs_Robertson = np.exp(1.93 + 0.5 * np.log(Qtn_inv) + 0.25 * np.log(fz) + 0.63 * Ic_inv)

### Calculate Zhang et al.

In [6]:
from National_Model_V1 import get_CPT

"""Enter the geology type of the site, the options are listed in geology value"""

geology = "Qal1"

a_values = [4.350665285788512,
            4.218856078544348,
            4.1648149061133575,
            4.15727678935075,
            4.191276553495072,
            4.216997360725642,
            4.300675095606855,
            4.368411533029867,
            4.30514634174693,
            4.5275854079196485,
            4.352834815858882]

geology_values = ['Beach and Dune',
                  'Floodplain',
                  'Glacial',
                  'Lagoonal',
                  'Valley Train',
                  'af/Qi',
                  'Qal1',
                  'Qal2',
                  'Qal3',
                  'Qoa',
                  'Qs']

a = a_values[geology_values.index(geology)]

fz, Ic, Qcncs, CPT_depth, Qcn_inv, Ic_inv, Qtn_Robertson, Ic_Robertson, kc = get_CPT(CPT_depth, CPT_qc, CPT_fs, dGWT)
index = (Qcncs > 0) & (fz > 0)  & (Ic < 4.0)

b = [0.24340982790022694, 2.4489890590703163, 2.4925075050951055]
f_Ic_constrain = 0.1 + 0.4 * (1 / (1 + np.exp(-b[1] * (Ic[index] - b[2]))))
global_velocity_qc = b[0] * np.log(Qcncs[index]) + f_Ic_constrain * np.log(fz[index]) + a

In [7]:
from SCPT_to_Vs_Tool_V2 import get_CPT, get_TT
ztop_TT, zbot_TT, DTT = get_TT(TT_depth, Meas_TT, source = 1.0)

Vsi = (zbot_TT - ztop_TT)/(DTT/1000)
# ztop_TT = ztop_TT[(Vsi > 50) & (Vsi < 1000)]
# zbot_TT = zbot_TT[(Vsi > 50) & (Vsi < 1000)]
# Vsi = Vsi[(Vsi > 50) & (Vsi < 1000)]


layer_ids = np.arange(len(Vsi))
lay_id_assign = np.full(len(CPT_depth), -1)


# Vectorized matching
for i in range(len(layer_ids)):

    mask = ((CPT_depth >= ztop_TT[i]) & (CPT_depth < zbot_TT[i]))

    lay_id_assign[mask] = layer_ids[i]

index = (lay_id_assign > -0.1) & (Qcncs > 0) & (fz > 0)  & (Ic < 4.0)

b = [0.258, 3.08, 2.47]
a = 4.20
f_Ic_constrain = 0.1 + 0.4 * (1 / (1 + np.exp(-b[1] * (Ic[index] - b[2]))))
global_velocity_single_a = b[0] * np.log(Qcncs[index]) + f_Ic_constrain * np.log(fz[index]) + a

cpt_slowness = 1 / np.exp(global_velocity_single_a)
dz = np.repeat(0.05, len(global_velocity_single_a))
layer_slowness = np.bincount(lay_id_assign[index], weights = cpt_slowness * dz)
layer_thickness = np.bincount(lay_id_assign[index], weights = dz)
layer_velocity_single_a = layer_thickness / layer_slowness

In [8]:
fz, Ic, Qtncs, CPT_depth, Qtn_inv, Ic_inv, Qtn_Robertson, Ic_Robertson, kc = get_CPT(CPT_depth, CPT_qt, CPT_fs, dGWT)
index = (Qtncs > 0) & (fz > 0)  & (Ic < 4.0)

b = [0.24340982790022694, 2.4489890590703163, 2.4925075050951055]
f_Ic_constrain = 0.1 + 0.4 * (1 / (1 + np.exp(-b[1] * (Ic[index] - b[2]))))
global_velocity_qt = b[0] * np.log(Qtncs[index]) + f_Ic_constrain * np.log(fz[index]) + a

### Calculate Vs Site Specific

In [9]:
def weighted_velocity (x , lay_id_assign_valid, Qtncs_CPT_valid, fz_CPT_valid, Ic_CPT_valid, Vsi):

    b = [0.24340982790022694, 2.4489890590703163, 2.4925075050951055]

    f_Ic_constrain = 0.1 + 0.4 * (1 / (1 + np.exp(-b[1] * (Ic_CPT_valid - b[2]))))

    cpt_velocity = b[0] * np.log(Qtncs_CPT_valid) + f_Ic_constrain * np.log(fz_CPT_valid) + x

    cpt_slowness = 1 / np.exp(cpt_velocity)
    dz = np.repeat(0.05, len(cpt_velocity))
    layer_slowness = np.bincount(lay_id_assign_valid, weights = cpt_slowness * dz)
    layer_thickness = np.bincount(lay_id_assign_valid, weights = dz)
    layer_velocity = layer_thickness / layer_slowness

    residual = np.log(Vsi) - np.log(layer_velocity)

    return residual

In [12]:
from SCPT_to_Vs_Tool_V2 import get_CPT, get_TT
from scipy.optimize import least_squares

fz, Ic, Qcncs, CPT_depth, Qtn_inv, Ic_inv, Qtn_Robertson, Ic_Robertson, kc = get_CPT(CPT_depth, CPT_qc, CPT_fs, dGWT)
ztop_TT, zbot_TT, DTT = get_TT(TT_depth, Meas_TT, source = 1.0)

Vsi = (zbot_TT - ztop_TT)/(DTT/1000)
# ztop_TT = ztop_TT[(Vsi > 50) & (Vsi < 1000)]
# zbot_TT = zbot_TT[(Vsi > 50) & (Vsi < 1000)]
# Vsi = Vsi[(Vsi > 50) & (Vsi < 1000)]


layer_ids = np.arange(len(Vsi))
lay_id_assign = np.full(len(CPT_depth), -1)


# Vectorized matching
for i in range(len(layer_ids)):

    mask = ((CPT_depth >= ztop_TT[i]) & (CPT_depth < zbot_TT[i]))

    lay_id_assign[mask] = layer_ids[i]

index = (lay_id_assign > -0.1) & (Qcncs > 0) & (fz > 0)  & (Ic < 4.0)

x = 5.0


result = least_squares(weighted_velocity, x, args=(lay_id_assign[index], Qcncs[index], fz[index], Ic[index], Vsi))

a = result.x[0]
b1 = 0.24340982790022694
b2 = 2.4489890590703163
b3 = 2.4925075050951055
print ("Optimized parameters: b1 =", b1, ", b2 =", b2, ", b3 =", b3, ", a =", a)

f_Ic_constrain = 0.1 + 0.4 * (1 / (1 + np.exp(-b2 * (Ic[index] - b3))))

velocity_ss = b1 * np.log(Qtncs[index]) + f_Ic_constrain * np.log(fz[index]) + a

cpt_slowness = 1 / np.exp(velocity_ss)
dz = np.repeat(0.05, len(velocity_ss))
layer_slowness = np.bincount(lay_id_assign[index], weights = cpt_slowness * dz)
layer_thickness = np.bincount(lay_id_assign[index], weights = dz)
layer_velocity = layer_thickness / layer_slowness

Optimized parameters: b1 = 0.24340982790022694 , b2 = 2.4489890590703163 , b3 = 2.4925075050951055 , a = 4.161087350413231


##### Export site specific results to CSV

In [14]:
# Export results to CSV
site_specific_results = pd.DataFrame({
    "depth (m)": CPT_depth[index],
    "velocity (m/s)": np.exp(velocity_ss)
})
site_specific_results.to_csv('Vs_ss_results.csv', index=False)

### GUI for all four methods

In [11]:
from SCPT_to_Vs_Tool_V2 import Master, interleave
import ipywidgets as widgets
from ipywidgets import HTML, Layout, HBox, VBox, Dropdown, FloatText, Button, Textarea
from IPython.display import display

%matplotlib widget

# Setup
style = {'description_width': 'initial'}

velocity_updated = []
top_depth_updated = []
bottom_depth_updated = []

VsZ_input = widgets.FloatText(value=1.0, description='VsZ Direct Interpretation (m/s)', style=style)
VsZ_glob_input = widgets.FloatText(value=1.0, description='VsZ Zhang et al. Model (m/s)', style=style)
VsZ_ss_input = widgets.FloatText(value=1.0, description='VsZ Zhang et al. Site Specific (m/s)', style=style)
VsZ_rob_input = widgets.FloatText(value=1.0, description='VsZ Robertson (2012) (m/s)', style=style)

button_layout = widgets.Layout(width='600px', height='40px')
EXPORT = Button(description='Export Result', layout = button_layout)
MANUAL_REVISE = Button(description='Manual Revision', layout = button_layout)

# Dynamic checkbox container (empty to start)
checkbox_box = widgets.VBox(layout=Layout(flex_flow='row wrap', width='100%'))

box_layout_2 = widgets.Layout(width='100%', border='solid 2px', padding='20px')
SEARCH = VBox([
    HTML('<center><font size="+1.5"><b>CPT-Vs Interpretation Tool</b>'),
    HBox([VsZ_input, VsZ_glob_input, VsZ_ss_input, VsZ_rob_input], layout=Layout(width='100%')),
    HBox([EXPORT, 
          MANUAL_REVISE], layout=Layout(width='100%')),
    HTML('<center><font size="+0.5">Slope Break Locations - Direct Interpretation'),
    checkbox_box  # add the checkboxes below the buttons
], layout=box_layout_2)

out = widgets.Output()

# --- Reactive SCPT update ---
def Update_SCPT(_):

    global velocity_updated
    global top_depth_updated
    global bottom_depth_updated

    out.clear_output()
    with out:

        velocity, TT, results, top_depth, bottom_depth = Master(TT_depth, Meas_TT, slope_break_method = 0)
        breaks = sorted(results.get('candidate_breakpoints', []))
        used_breaks = [str(round(b, 3)) for b in results.get('breaks', [])]

        # using subplot to plot qt, fz, Ic, Vs with depth for the selected traveltimeMeta_ID
        fig, axs = plt.subplots(1, 6, figsize=(13, 5), dpi=300)

        # Qtn Plot - philosophy is to show robertson (Qtn), Qcn(inv) and Qcncs showing the Kc correction effects
        axs[0].plot(Qtn_Robertson, CPT_depth, label='Qtn', color = 'grey', linewidth = 1.2)
        # axs[0].plot(Qcn_inv, CPT_depth, label='Qcn_inv', color='red', linewidth = 1.2)
        axs[0].plot(Qcncs[index], CPT_depth[index], label='Qtncs', color='tab:blue', linewidth = 1.2) # Updated: Due to the similar results from the comparison work, Qtncs is used
        axs[0].set_xlim(0, 1000)
        axs[0].set_xticks(np.arange(0, 1001, 200))
        axs[0].grid(True, which="both", ls="--")
        axs[0].set_xlabel('Qtn, Qtn,cs') #temporary changed into Qtn 
        axs[0].set_ylabel('Depth (m)')
        # axs[0].set_title('Qcn, Qcn,cs vs Depth')
        axs[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        # add label "(a)" on the top right corner of the first subplot
        axs[0].text(0.95, 0.93, '(a)', transform=axs[0].transAxes, fontsize=12, verticalalignment='bottom', horizontalalignment='right')

        # σv' Plot
        axs[1].plot((0, fz[0]), (0, CPT_depth[0]), color='tab:blue', linestyle='dashed', linewidth = 1.2)
        axs[1].plot(fz, CPT_depth, label='$\\sigma_v^\\prime$', color='tab:blue', linewidth = 1.2)
        if np.max(fz) > 4:
            axs[1].set_xlim(0, 5)
            axs[1].set_xticks(np.arange(0, 6, 1))
        elif np.max(fz) > 3:
            axs[1].set_xlim(0, 4)
            axs[1].set_xticks(np.arange(0, 5, 1))
        else:
            axs[1].set_xlim(0, 3)
            axs[1].set_xticks(np.arange(0, 4, 1))
        axs[1].grid(True, which="both", ls="--")
        axs[1].set_xlabel('$\\sigma_v^\\prime$ (atm)')
        # axs[1].set_title('$\\sigma_v^\\prime$ vs Depth')
        axs[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        axs[1].text(0.95, 0.93, '(b)', transform=axs[1].transAxes, fontsize=12, verticalalignment='bottom', horizontalalignment='right')


        axs[2].plot(Ic_Robertson, CPT_depth, label='Ic')
        # axs[2].plot(Ic[index], CPT_depth[index], label='Ic_inv', color = 'black') # hide for now for the NCEE plots
        axs[2].set_xlim(1, 4)
        axs[2].set_xlabel('Ic')
        # axs[2].set_title('Ic vs Depth')
        axs[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        axs[2].grid(True, which="both", ls="--")
        axs[2].text(0.95, 0.93, '(c)', transform=axs[2].transAxes, fontsize=12, verticalalignment='bottom', horizontalalignment='right')

        # axs[3].plot(interleave(np.exp(Vs_pred_plot), np.exp(Vs_pred_plot)), interleave(ztop_plot, zbot_plot), label='Vsi_Model1', color='green')
        # axs[3].plot(np.exp(Vs_plot), CPT_depth_plot, label='Vs_model', color='orange')
        axs[3].plot(interleave(Vsi, Vsi), interleave(ztop_TT, zbot_TT), label='Measured', color='black')
        axs[3].plot(interleave(layer_velocity_single_a, layer_velocity_single_a), interleave(ztop_TT, zbot_TT), label='Spatially Ergodic', color='tab:blue')
        axs[3].plot(interleave(layer_velocity, layer_velocity), interleave(ztop_TT, zbot_TT), label='Site Specific', color='tab:orange')
        axs[3].text(0.95, 0.93, '(d)', transform=axs[3].transAxes, fontsize=12, verticalalignment='bottom', horizontalalignment='right')


        if np.max([np.max(Vsi), np.max(layer_velocity)]) > 800:
            axs[3].set_xlim(0, 1200)
            axs[3].set_xticks(np.arange(0, 1201, 200))
        elif np.max([np.max(Vsi), np.max(layer_velocity)]) > 600:
            axs[3].set_xlim(0, 800)
            axs[3].set_xticks(np.arange(0, 801, 200))
        else:  
            axs[3].set_xlim(0, 600)
            axs[3].set_xticks(np.arange(0, 601, 100))
        axs[3].set_xlabel('Interval Velocity (m/s)')
        # axs[3].set_title('Interval Velocity vs Depth')
        axs[3].grid(True, which="both", ls="--")
        axs[3].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

        axs[4].scatter(TT, TT_depth, label='Travel Time Data', color='black', s=10)

        colors = plt.get_cmap('tab10').colors

        # Plot each segment with a different color
        for i in range(len(results['breaks']) - 1):
            # Define segment depth range
            z_start = results['breaks'][i]
            z_end = results['breaks'][i + 1]

            # Get depths in this segment (new_depths ensures breaks are included)
            segment_mask = (results['new_depths'] >= z_start) & (results['new_depths'] <= z_end)
            segment_depths = results['new_depths'][segment_mask]
            segment_tt = results['fitted_values_for_plot'][segment_mask]

            # Plot segment
            axs[4].plot(segment_tt, segment_depths, color=colors[i])

        axs[4].set_xlabel('Travel Time (ms)')
        axs[4].set_xlim(0, np.max(TT) * 1.1)
        if np.max(TT) > 150:
            axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 50))
        elif np.max(TT) > 90:
            axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 30))
        elif np.max(TT) > 50:
            axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 20))
        else:
            axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 10))
        # axs[4].set_title('Travel Time vs Depth')
        axs[4].grid(True, which="both", ls="--")
        axs[4].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        axs[4].text(0.95, 0.93, '(e)', transform=axs[4].transAxes, fontsize=12, verticalalignment='bottom', horizontalalignment='right')

        # axs[5].plot(Vs_Robertson, CPT_depth, color='grey', label = 'Robertson (2012)', alpha=0.5)
        axs[5].plot(np.exp(global_velocity_single_a), CPT_depth[index], label='Zhang et al. (Spatially Ergodic)', color='tab:blue') # Overall model
        # axs[5].plot(np.exp(global_velocity_qc), CPT_depth, label='Zhang et al.', color='tab:blue')
        axs[5].plot(np.exp(velocity_ss), CPT_depth[index], label='Zhang et al. (Site Specific)', color='tab:orange')
        axs[5].plot(interleave(velocity, velocity), interleave(top_depth, bottom_depth), label='Layer-Based Regression', color='black')

        if np.max([np.max(np.exp(global_velocity_qc)), np.max(np.exp(velocity_ss)), np.max(velocity)]) > 800:
            axs[5].set_xlim(0, 1000)
            axs[5].set_xticks(np.arange(0, 1001, 200))
        elif np.max([np.max(np.exp(global_velocity_qc)), np.max(np.exp(velocity_ss)), np.max(velocity)]) > 600:
            axs[5].set_xlim(0, 800)
            axs[5].set_xticks(np.arange(0, 801, 200))
        else:
            axs[5].set_xlim(0, 600)
            axs[5].set_xticks(np.arange(0, 601, 100))

        axs[5].set_xlabel('Velocity (m/s)')
        # axs[5].set_title('Velocity vs Depth')
        axs[5].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        axs[5].grid(True, which="both", ls="--")
        axs[5].text(0.95, 0.93, '(f)', transform=axs[5].transAxes, fontsize=12, verticalalignment='bottom', horizontalalignment='right')


        # Share common y-axis limits across all subplots
        y_min = np.nanmin([np.nanmin(CPT_depth), np.nanmin(ztop_TT), np.nanmin(TT_depth)])
        y_max = np.nanmax([np.nanmax(CPT_depth), np.nanmax(zbot_TT), np.nanmax(TT_depth)])
        for ax in axs:
            ax.set_ylim(y_max + 1, 0)  # inverted depth axis, same range for all

        plt.subplots_adjust(bottom=0.28, wspace=0.35)
        plt.tight_layout()
        plt.show()

        fig.savefig('CPT_Vs_Interpretation.png', dpi=300, bbox_inches='tight')

        # Calculate VsZ for direct interpretation
        top_depth_dir = np.asarray(top_depth)
        top_depth_dir[0] = 0.0
        VsZ = np.max(np.asarray(bottom_depth)) / np.sum((np.asarray(bottom_depth) - top_depth_dir) / np.asarray(velocity))

        # Calculate VsZ for Robertson
        VsZ_robertson_top_depth = np.insert(np.asarray(CPT_depth[:-1]), 0, 0.0)
        VsZ_robertson = np.max(CPT_depth) / np.sum((np.asarray(CPT_depth) - VsZ_robertson_top_depth) / np.asarray(Vs_Robertson))

        # Calculate VsZ for national model
        VsZ_glob_top_depth = np.insert(np.asarray(CPT_depth[index][:-1]), 0, 0.0)
        VsZ_glob = np.max(CPT_depth[index]) / np.sum((np.asarray(CPT_depth[index]) - VsZ_glob_top_depth) / np.asarray(np.exp(global_velocity_qc[index])))

        # Calculate VsZ for global model
        VsZ_ss_top_depth = np.insert(np.asarray(CPT_depth[index][:-1]), 0, 0.0)
        VsZ_ss = np.max(CPT_depth[index]) / np.sum((np.asarray(CPT_depth[index]) - VsZ_ss_top_depth) / np.asarray(np.exp(velocity_ss)))

        # Update the checkboxes dynamically
        new_checkboxes = [
            widgets.Checkbox(description=str(round(b, 3)), indent=False, value=(str(round(b, 3)) in used_breaks))
            for b in sorted(breaks)
        ]
        checkbox_box.children = new_checkboxes

        VsZ_input.value = np.round(VsZ, 0)
        VsZ_glob_input.value = np.round(VsZ_glob, 0)
        VsZ_ss_input.value = np.round(VsZ_ss, 0)
        VsZ_rob_input.value = np.round(VsZ_robertson, 0)

        velocity_updated = velocity
        top_depth_updated = top_depth
        bottom_depth_updated = bottom_depth

def Full_Manual(_):

    global velocity_updated
    global top_depth_updated
    global bottom_depth_updated

    out.clear_output()
    with out:
        # Get selected breakpoints
        selected_breakpoints = [float(checkbox.description) for checkbox in checkbox_box.children if checkbox.value]
        if not selected_breakpoints:
            print("No breakpoints selected.")
            return
        # Run full manual regression
        velocity, TT, results, top_depth, bottom_depth = Master(TT_depth, Meas_TT, slope_break_method=1, breakpoints=selected_breakpoints)
                # using subplot to plot qt, fz, Ic, Vs with depth for the selected traveltimeMeta_ID
        fig, axs = plt.subplots(1, 6, figsize=(13, 5), dpi=300)
        # Qtn Plot - philosophy is to show robertson (Qtn), Qcn(inv) and Qcncs showing the Kc correction effects
        axs[0].plot(Qtn_Robertson, CPT_depth, label='Qtn', color = 'grey', linewidth = 1.2)
        # axs[0].plot(Qcn_inv, CPT_depth, label='Qcn_inv', color='red', linewidth = 1.2)
        axs[0].plot(Qcncs[index], CPT_depth[index], label='Qtncs', color='tab:blue', linewidth = 1.2) # Updated: Due to the similar results from the comparison work, Qtncs is used
        axs[0].set_xlim(0, 1000)
        axs[0].set_xticks(np.arange(0, 1001, 200))
        axs[0].grid(True, which="both", ls="--")
        axs[0].set_xlabel('Qtn, Qtn,cs') #temporary changed into Qtn 
        axs[0].set_ylabel('Depth (m)')
        # axs[0].set_title('Qcn, Qcn,cs vs Depth')
        axs[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        # add label "(a)" on the top right corner of the first subplot
        axs[0].text(0.95, 0.93, '(a)', transform=axs[0].transAxes, fontsize=12, verticalalignment='bottom', horizontalalignment='right')

        # σv' Plot
        axs[1].plot((0, fz[0]), (0, CPT_depth[0]), color='tab:blue', linestyle='dashed', linewidth = 1.2)
        axs[1].plot(fz, CPT_depth, label='$\\sigma_v^\\prime$', color='tab:blue', linewidth = 1.2)
        if np.max(fz) > 4:
            axs[1].set_xlim(0, 5)
            axs[1].set_xticks(np.arange(0, 6, 1))
        elif np.max(fz) > 3:
            axs[1].set_xlim(0, 4)
            axs[1].set_xticks(np.arange(0, 5, 1))
        else:
            axs[1].set_xlim(0, 3)
            axs[1].set_xticks(np.arange(0, 4, 1))
        axs[1].grid(True, which="both", ls="--")
        axs[1].set_xlabel('$\\sigma_v^\\prime$ (atm)')
        # axs[1].set_title('$\\sigma_v^\\prime$ vs Depth')
        axs[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        axs[1].text(0.95, 0.93, '(b)', transform=axs[1].transAxes, fontsize=12, verticalalignment='bottom', horizontalalignment='right')


        axs[2].plot(Ic_Robertson, CPT_depth, label='Ic')
        # axs[2].plot(Ic[index], CPT_depth[index], label='Ic_inv', color = 'black') # hide for now for the NCEE plots
        axs[2].set_xlim(1, 4)
        axs[2].set_xlabel('Ic')
        # axs[2].set_title('Ic vs Depth')
        axs[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        axs[2].grid(True, which="both", ls="--")
        axs[2].text(0.95, 0.93, '(c)', transform=axs[2].transAxes, fontsize=12, verticalalignment='bottom', horizontalalignment='right')

        # axs[3].plot(interleave(np.exp(Vs_pred_plot), np.exp(Vs_pred_plot)), interleave(ztop_plot, zbot_plot), label='Vsi_Model1', color='green')
        # axs[3].plot(np.exp(Vs_plot), CPT_depth_plot, label='Vs_model', color='orange')
        axs[3].plot(interleave(Vsi, Vsi), interleave(ztop_TT, zbot_TT), label='Measured', color='black')
        axs[3].plot(interleave(layer_velocity_single_a, layer_velocity_single_a), interleave(ztop_TT, zbot_TT), label='Spatially Ergodic', color='tab:blue')
        axs[3].plot(interleave(layer_velocity, layer_velocity), interleave(ztop_TT, zbot_TT), label='Site Specific', color='tab:orange')
        axs[3].text(0.95, 0.93, '(d)', transform=axs[3].transAxes, fontsize=12, verticalalignment='bottom', horizontalalignment='right')


        if np.max([np.max(Vsi), np.max(layer_velocity)]) > 800:
            axs[3].set_xlim(0, 1200)
            axs[3].set_xticks(np.arange(0, 1201, 200))
        elif np.max([np.max(Vsi), np.max(layer_velocity)]) > 600:
            axs[3].set_xlim(0, 800)
            axs[3].set_xticks(np.arange(0, 801, 200))
        else:  
            axs[3].set_xlim(0, 600)
            axs[3].set_xticks(np.arange(0, 601, 100))
        axs[3].set_xlabel('Interval Velocity (m/s)')
        # axs[3].set_title('Interval Velocity vs Depth')
        axs[3].grid(True, which="both", ls="--")
        axs[3].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

        axs[4].scatter(TT, TT_depth, label='Travel Time Data', color='black', s=10)

        colors = plt.get_cmap('tab10').colors

        # Plot each segment with a different color
        for i in range(len(results['breaks']) - 1):
            # Define segment depth range
            z_start = results['breaks'][i]
            z_end = results['breaks'][i + 1]

            # Get depths in this segment (new_depths ensures breaks are included)
            segment_mask = (results['new_depths'] >= z_start) & (results['new_depths'] <= z_end)
            segment_depths = results['new_depths'][segment_mask]
            segment_tt = results['fitted_values_for_plot'][segment_mask]

            # Plot segment
            axs[4].plot(segment_tt, segment_depths, color=colors[i])

        axs[4].set_xlabel('Travel Time (ms)')
        axs[4].set_xlim(0, np.max(TT) * 1.1)
        if np.max(TT) > 150:
            axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 50))
        elif np.max(TT) > 90:
            axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 30))
        elif np.max(TT) > 50:
            axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 20))
        else:
            axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 10))
        # axs[4].set_title('Travel Time vs Depth')
        axs[4].grid(True, which="both", ls="--")
        axs[4].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        axs[4].text(0.95, 0.93, '(e)', transform=axs[4].transAxes, fontsize=12, verticalalignment='bottom', horizontalalignment='right')

        axs[5].plot(Vs_Robertson, CPT_depth, color='grey', label = 'Robertson (2012)', alpha=0.5)
        axs[5].plot(np.exp(global_velocity_single_a), CPT_depth[index], label='Zhang et al. (Spatially Ergodic)', color='tab:blue') # Overall model
        # axs[5].plot(np.exp(global_velocity_qc), CPT_depth, label='Zhang et al.', color='tab:blue')
        axs[5].plot(np.exp(velocity_ss), CPT_depth[index], label='Zhang et al. (Site Specific)', color='tab:orange')
        axs[5].plot(interleave(velocity, velocity), interleave(top_depth, bottom_depth), label='Layer-Based Regression', color='black')

        if np.max([np.max(np.exp(global_velocity_qc)), np.max(np.exp(velocity_ss)), np.max(velocity)]) > 800:
            axs[5].set_xlim(0, 1000)
            axs[5].set_xticks(np.arange(0, 1001, 200))
        elif np.max([np.max(np.exp(global_velocity_qc)), np.max(np.exp(velocity_ss)), np.max(velocity)]) > 600:
            axs[5].set_xlim(0, 800)
            axs[5].set_xticks(np.arange(0, 801, 200))
        else:
            axs[5].set_xlim(0, 600)
            axs[5].set_xticks(np.arange(0, 601, 100))

        axs[5].set_xlabel('Velocity (m/s)')
        # axs[5].set_title('Velocity vs Depth')
        axs[5].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        axs[5].grid(True, which="both", ls="--")
        axs[5].text(0.95, 0.93, '(f)', transform=axs[5].transAxes, fontsize=12, verticalalignment='bottom', horizontalalignment='right')


        # Share common y-axis limits across all subplots
        y_min = np.nanmin([np.nanmin(CPT_depth), np.nanmin(ztop_TT), np.nanmin(TT_depth)])
        y_max = np.nanmax([np.nanmax(CPT_depth), np.nanmax(zbot_TT), np.nanmax(TT_depth)])
        for ax in axs:
            ax.set_ylim(y_max + 1, 0)  # inverted depth axis, same range for all

        plt.subplots_adjust(bottom=0.28, wspace=0.35)
        plt.tight_layout()
        plt.show()

        # save fig as a PNG file
        fig.savefig('CPT_Vs_Interpretation.png', dpi=300, bbox_inches='tight')

        # Calculate VsZ for direct interpretation
        top_depth_dir = np.asarray(top_depth)
        top_depth_dir[0] = 0.0
        VsZ = np.max(np.asarray(bottom_depth)) / np.sum((np.asarray(bottom_depth) - top_depth_dir) / np.asarray(velocity))

        # Calculate VsZ for Robertson
        VsZ_robertson_top_depth = np.insert(np.asarray(CPT_depth[:-1]), 0, 0.0)
        VsZ_robertson = np.max(CPT_depth) / np.sum((np.asarray(CPT_depth) - VsZ_robertson_top_depth) / np.asarray(Vs_Robertson))

        # Calculate VsZ for national model
        VsZ_glob_top_depth = np.insert(np.asarray(CPT_depth[index][:-1]), 0, 0.0)
        VsZ_glob = np.max(CPT_depth[index]) / np.sum((np.asarray(CPT_depth[index]) - VsZ_glob_top_depth) / np.asarray(np.exp(global_velocity_qc[index])))

        # Calculate VsZ for global model
        VsZ_ss_top_depth = np.insert(np.asarray(CPT_depth[index][:-1]), 0, 0.0)
        VsZ_ss = np.max(CPT_depth[index]) / np.sum((np.asarray(CPT_depth[index]) - VsZ_ss_top_depth) / np.asarray(np.exp(velocity_ss)))

        VsZ_input.value = np.round(VsZ, 0)
        VsZ_glob_input.value = np.round(VsZ_glob, 0)
        VsZ_ss_input.value = np.round(VsZ_ss, 0)
        VsZ_rob_input.value = np.round(VsZ_robertson,0) 

        velocity_updated = velocity
        top_depth_updated = top_depth
        bottom_depth_updated = bottom_depth

# Export Function
def Export (_):
    
    global velocity_updated
    global top_depth_updated
    global bottom_depth_updated

    # Save to CSV
    try:
        VS_SLOPEBREAK_DF = pd.read_csv('SLOPEBREAK_Vs.csv')
    except FileNotFoundError:
        VS_SLOPEBREAK_DF = pd.DataFrame(columns=['travelTimeMeta_ID', 'Velocity', 'Top_depth', 'Bottom_depth'])

    velocity_save = velocity_updated
    top_depth_save = top_depth_updated
    bottom_depth_save = bottom_depth_updated

    # Save the Vs profile to a CSV file
    new_data2 = pd.DataFrame({
    'Velocity': velocity_save,
    'Top_depth': top_depth_save,
    'Bottom_depth': bottom_depth_save})
    VS_SLOPEBREAK_DF = pd.concat([VS_SLOPEBREAK_DF, new_data2], ignore_index=True)
    # Save to CSV
    VS_SLOPEBREAK_DF.to_csv('SLOPEBREAK_Vs.csv', index=False)

# Initial display
display(SEARCH)
display(out)

# Trigger first update manually
Update_SCPT(None)

# Attach trigger to dropdown
EXPORT.on_click(Export)
MANUAL_REVISE.on_click(Full_Manual)

Output()